In [11]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

In [12]:
df = pd.read_csv('laptops_Price.csv')

In [13]:
df.head()

,Name,Price,Review,Description
0,Asus VivoBook...,$295.99,14,"Asus VivoBook X441NA-GA190 Chocolate Black, 14..."
1,Prestigio Smar...,$299.00,8,"Prestigio SmartBook 133S Dark Grey, 13.3"" FHD ..."
2,Prestigio Smar...,$299.00,12,"Prestigio SmartBook 133S Gold, 13.3"" FHD IPS, ..."
3,Aspire E1-510,$306.99,2,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux"
4,Lenovo V110-15...,$321.94,5,"Lenovo V110-15IAP, 15.6"" HD, Celeron N3350 1.1..."


In [14]:
#Data Cleaning
df["Price"] = df["Price"].str.replace("$", "").str.replace(",", "").astype(float)

In [15]:
df["RAM"] = df["Description"].str.extract(r"(\d+)\s*gb")[0].astype(float)

In [16]:
df.head()

,Name,Price,Review,Description,RAM
0,Asus VivoBook...,295.99,14,"Asus VivoBook X441NA-GA190 Chocolate Black, 14...",NaN
1,Prestigio Smar...,299.00,8,"Prestigio SmartBook 133S Dark Grey, 13.3"" FHD ...",NaN
2,Prestigio Smar...,299.00,12,"Prestigio SmartBook 133S Gold, 13.3"" FHD IPS, ...",NaN
3,Aspire E1-510,306.99,2,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux",NaN
4,Lenovo V110-15...,321.94,5,"Lenovo V110-15IAP, 15.6"" HD, Celeron N3350 1.1...",NaN


In [17]:
def extract_storage(text):
    m = re.search(r"(\d+)\s*gb\s*(ssd|hdd|emmc|nvme|storage)?", text)
    if m:
        return float(m.group(1))
    m = re.search(r"(\d+)\s*tb", text)
    if m:
        return float(m.group(1)) * 1024
    return np.nan

In [18]:
df.head()

,Name,Price,Review,Description,RAM
0,Asus VivoBook...,295.99,14,"Asus VivoBook X441NA-GA190 Chocolate Black, 14...",NaN
1,Prestigio Smar...,299.00,8,"Prestigio SmartBook 133S Dark Grey, 13.3"" FHD ...",NaN
2,Prestigio Smar...,299.00,12,"Prestigio SmartBook 133S Gold, 13.3"" FHD IPS, ...",NaN
3,Aspire E1-510,306.99,2,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux",NaN
4,Lenovo V110-15...,321.94,5,"Lenovo V110-15IAP, 15.6"" HD, Celeron N3350 1.1...",NaN


In [20]:
df["Has_SSD"]    = df["Description"].str.contains("ssd|nvme").astype(int)   # 1 ya 0
df["Screen_inch"]= df["Description"].str.extract(r"(\d+\.?\d*)[\"']")[0]   # 15.6, 13.3 etc
df["Is_Windows"] = df["Description"].str.contains("windows").astype(int)
df["Is_Core_i"]  = df["Description"].str.contains(r"core\s*i\d").astype(int)
df["Is_Ryzen"]   = df["Description"].str.contains("ryzen").astype(int)

In [21]:
df.head()

,Name,Price,Review,Description,RAM,Has_SSD,Screen_inch,Is_Windows,Is_Core_i,Is_Ryzen
0,Asus VivoBook...,295.99,14,"Asus VivoBook X441NA-GA190 Chocolate Black, 14...",NaN,0,14,0,0,0
1,Prestigio Smar...,299.00,8,"Prestigio SmartBook 133S Dark Grey, 13.3"" FHD ...",NaN,0,13.3,0,0,0
2,Prestigio Smar...,299.00,12,"Prestigio SmartBook 133S Gold, 13.3"" FHD IPS, ...",NaN,0,13.3,0,0,0
3,Aspire E1-510,306.99,2,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux",NaN,0,15.6,0,0,0
4,Lenovo V110-15...,321.94,5,"Lenovo V110-15IAP, 15.6"" HD, Celeron N3350 1.1...",NaN,0,15.6,0,0,0


In [22]:
df["Brand"] = df["Name"].str.split().str[0]  # "Lenovo V110..." → "Lenovo"
le = LabelEncoder()
df["Brand_enc"] = le.fit_transform(df["Brand"])  # "Lenovo"→3, "Asus"→1 etc

In [23]:
df.head()

,Name,Price,Review,Description,RAM,Has_SSD,Screen_inch,Is_Windows,Is_Core_i,Is_Ryzen,Brand,Brand_enc
0,Asus VivoBook...,295.99,14,"Asus VivoBook X441NA-GA190 Chocolate Black, 14...",NaN,0,14,0,0,0,Asus,3
1,Prestigio Smar...,299.00,8,"Prestigio SmartBook 133S Dark Grey, 13.3"" FHD ...",NaN,0,13.3,0,0,0,Prestigio,12
2,Prestigio Smar...,299.00,12,"Prestigio SmartBook 133S Gold, 13.3"" FHD IPS, ...",NaN,0,13.3,0,0,0,Prestigio,12
3,Aspire E1-510,306.99,2,"15.6"", Pentium N3520 2.16GHz, 4GB, 500GB, Linux",NaN,0,15.6,0,0,0,Aspire,2
4,Lenovo V110-15...,321.94,5,"Lenovo V110-15IAP, 15.6"" HD, Celeron N3350 1.1...",NaN,0,15.6,0,0,0,Lenovo,8


In [ ]:
df["Storage_GB"] = df["Description"].apply(extract_storage")

features = ["RAM", "Storage_GB", "Has_SSD", "Screen_inch",
            "Is_Windows", "Is_Core_i", "Is_Ryzen", "Brand_enc", "Review"]
X = df[features]   # input — yeh sab deke price predict karenge
y = df["Price_num"] # output — yahi predict karna hai

KeyError: "['Storage_GB'] not in index"